Part 1 : Data Import

In [2]:
import pandas as pd
import numpy as np
from scipy import stats


# Load the dataset into a pandas DataFrame
try:
    data = pd.read_csv('data.csv', sep='\t')
    print("Dataset has been successfully loaded.")
except Exception as e:
    print(f"Failed to load data: {e}")


# Display the first few rows of the dataset
print(data.head())
# Separate numeric columns from non-numeric columns
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
non_numeric_cols = data.select_dtypes(exclude=['int64', 'float64']).columns

/tmp/ipykernel_1418/1550843614.py:8: DtypeWarning: Columns (0,2,4,5,6,8,9,10,12,13,14,16,17,18,20,21,22,24,25,26,28,29,30,32,33,34,36,37,38,40,41,42,44,45,46,48,49,50,52,53,54,56,57,58,60,61,62,64,65,66,68,69,70,72,73,74,76,77,78,80,81,82,84,85,86,88,89,90,92,93,94,96,97,98,100,101,102,104,105,106,108,109,110,112,113,114,116,117,118,120,121,122,124,125,126,128,129) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('data.csv', sep='\t')


Dataset has been successfully loaded.
  Unnamed: 0        date     id size_grp       age   aliq_at aliq_mat  \
0      53170   9/30/1962  33814     mega -0.410891  0.343616      0.0   
1      53907  10/31/1962  33814     mega  -0.41374  0.333333      0.0   
2      54659  11/30/1962  33814     mega -0.401786  0.331697      0.0   
3      55425  12/31/1962  33814     mega  -0.37069  0.322252      0.0   
4      56207   1/31/1963  33814     mega -0.367951  0.324561      0.0   

   ami_126d     at_be    at_gr1  ... turnover_var_126d  z_score  \
0 -0.290922 -0.447445  0.247967  ...          0.274763      0.0   
1 -0.281893 -0.447598  0.231343  ...          0.210598      0.0   
2 -0.297587 -0.448791  0.221485  ...          0.167995      0.0   
3 -0.309585 -0.439808  0.215019  ...           0.12516      0.0   
4 -0.308729 -0.441892  0.210828  ...          0.035714      0.0   

  zero_trades_126d zero_trades_21d zero_trades_252d       ret Unnamed: 126  \
0         0.178426        0.033875        

Part 2 :Sanity Checks

In [3]:
# Check for constant columns
constant_cols = numeric_cols[data[numeric_cols].std() == 0]
print("Constant columns:")
print(constant_cols)

# Check for duplicate rows
duplicate_rows = data.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows}")


# Check for non-numeric values in numeric columns
for col in numeric_cols:
    if data[col].isnull().any():
        print(f"Column {col} contains null values.")
    else:
        print(f"Column {col} does not contain null values.")

# Check date format consistency
date_col = 'date'  # Replace 'date' with the actual column name
try:
    pd.to_datetime(data[date_col])
except ValueError:
    print("Date formats are not consistent.")
else:
    print("Date formats appear to be consistent.")

# Check for extreme values (highly skewed distributions)

for col in numeric_cols:
    skewness = stats.skew(data[col])
    if abs(skewness) > 2:
        print(f"Column {col} has a highly skewed distribution (skewness: {skewness}).")
    else:
        print(f"Column {col} does not have a highly skewed distribution (skewness: {skewness}).")

Constant columns:
Index(['Unnamed: 127'], dtype='object')
Duplicate rows: 0
Column ami_126d contains null values.
Column at_turnover contains null values.
Column beta_dimson_21d contains null values.
Column bidaskhl_21d contains null values.
Column coa_gr1a contains null values.
Column corr_1260d contains null values.
Column debt_me contains null values.
Column dolvol_var_126d contains null values.
Column ebitda_mev contains null values.
Column eqnpo_12m contains null values.
Column fnl_gr1a contains null values.
Column iskew_capm_21d contains null values.
Column ivol_capm_252d contains null values.
Column lnoa_gr1a contains null values.
Column mispricing_perf contains null values.
Column netis_at contains null values.
Column niq_at contains null values.
Column noa_at contains null values.
Column oaccruals_ni contains null values.
Column op_at contains null values.
Column ppeinv_gr1a contains null values.
Column qmj_safety contains null values.
Column ret_12_1 contains null values.
Col

Part 3 : Clean Dataset

In [4]:
# Create a copy of the original data
cleanedData = data.copy()

# Remove constant columns
constant_cols = [col for col in cleanedData.columns if cleanedData[col].nunique() == 1]
cleanedData = cleanedData.drop(constant_cols, axis=1)

# Update numeric_cols to exclude removed columns
numeric_cols = [col for col in numeric_cols if col in cleanedData.columns]

# Handle duplicate rows (not necessary in this case since there are no duplicates)
# cleanedData = cleanedData.drop_duplicates()

# Handle non-numeric values in numeric columns
for col in numeric_cols:
    if cleanedData[col].isnull().any():
        # Fill null values with the mean of the column
        cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mean())
    else:
        print(f"Column {col} does not contain null values.")

# Check for NaN values in the dataset
nan_cols = cleanedData.columns[cleanedData.isnull().any()].tolist()
print("Columns with NaN values:", nan_cols)

# Fill NaN values with the mean of the column (for numerical columns)
for col in nan_cols:
    if cleanedData[col].dtype.kind in 'bifc':  # Check if column is numeric
        if not cleanedData[col].empty:
            cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mean())
        else:
            print(f"Column {col} is empty and cannot be filled with the mean.")
    else:
        # Fill NaN values with the mode of the column (for categorical columns)
        if not cleanedData[col].empty:
            cleanedData[col] = cleanedData[col].fillna(cleanedData[col].mode().iloc[0])
        else:
            print(f"Column {col} is empty and cannot be filled with the mode.")

# Check if there are still NaN values in the dataset
if cleanedData.isnull().values.any():
    print("There are still NaN values in the dataset.")
    nan_cols = cleanedData.columns[cleanedData.isnull().any()].tolist()
    print("Columns with NaN values:", nan_cols)
else:
    print("All NaN values have been filled.")

# Check date format consistency
date_col = 'date'  # Replace 'date' with the actual column name
try:
    cleanedData[date_col] = pd.to_datetime(cleanedData[date_col])
except ValueError:
    print("Date formats are not consistent.")
except KeyError:
    print("Date column not found.")
else:
    print("Date formats appear to be consistent.")

# Handle extreme values (highly skewed distributions)
for col in numeric_cols:
    skewness = stats.skew(cleanedData[col])
    if abs(skewness) > 2:
        # Apply logarithmic transformation to reduce skewness (using log1 to avoid log(0) errors)
        cleanedData[col] = np.log1p(cleanedData[col])
    else:
        print(f"Column {col} does not have a highly skewed distribution (skewness: {skewness}).")

print(cleanedData.head())

Columns with NaN values: ['Unnamed: 0', 'date', 'id', 'size_grp', 'age', 'aliq_at', 'aliq_mat', 'at_be', 'at_gr1', 'at_me', 'be_gr1a', 'be_me', 'beta_60m', 'betabab_1260d', 'betadown_252d', 'bev_mev', 'capx_gr1', 'cash_at', 'chcsho_12m', 'col_gr1a', 'cop_at', 'cop_atl1', 'coskew_21d', 'cowc_gr1a', 'dbnetis_at', 'dgp_dsale', 'div12m_me', 'dolvol_126d', 'dsale_drec', 'ebit_bev', 'ebit_sale', 'emp_gr1', 'eq_dur', 'eqnetis_at', 'eqnpo_me', 'eqpo_me', 'fcf_me', 'gp_at', 'gp_atl1', 'inv_gr1a', 'iskew_ff3_21d', 'iskew_hxz4_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'ivol_hxz4_21d', 'kz_index', 'lti_gr1a', 'market_equity', 'mispricing_mgmt', 'ncoa_gr1a', 'ncol_gr1a', 'netdebt_me', 'nfna_gr1a', 'ni_be', 'ni_me', 'niq_at_chg1', 'niq_be', 'nncoa_gr1a', 'noa_gr1a', 'o_score', 'oaccruals_at', 'ocf_at', 'ocf_at_chg1', 'ocf_me', 'op_atl1', 'ope_be', 'opex_at', 'prc', 'prc_highprc_252d', 'qmj_prof', 'resff3_12_1', 'resff3_6_1', 'ret_1_0', 'ret_12_7', 'ret_3_1', 'ret_6_1', 'ret_9_1', 'rmax1_21d', 'rmax5_21

/home/Loris/miniconda3/envs/MLFIN/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
